In [1]:
import ee
ee.Initialize(project="ee-mnolasco")
BASE = "users/mnolasco/FORMAR_2024_2026"
COL_SALIDA = f"{BASE}/mascaras_s2_60m"
try:
    ee.data.createAsset({"type": "IMAGE_COLLECTION"}, COL_SALIDA)
except Exception as e:
    print("Colección:", str(e)[:80])          # si ya existe, avisa y sigue

prov = ee.FeatureCollection("projects/ee-mnolasco/assets/Provincias")
deptos = ee.FeatureCollection(f"{BASE}/departments_selected")
PROVINCIAS = {"Buenos Aires": 24, "Córdoba": 13, "Entre Ríos": 14, "La Pampa": 22, "Santa Fe": 4}

# Etiquetas: agrícola estable en > 14 de las 22 campañas MODIS (2000/01–2021/22)
modis_hist = [ee.Image(f"projects/ee-mnolasco/assets/Mapas_binarios_MODIS/{a}-{a + 1}_mascara_cultivos_3M_DEM1000")
              for a in range(2000, 2022)]
etiqueta = ee.ImageCollection(modis_hist).sum().gt(14).rename("b1")

B_S2 = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
NOMBRES = ["BLUE", "GREEN", "RED", "RED_EDGE1", "RED_EDGE2", "RED_EDGE3", "NIR", "RED_EDGE4", "SWIR1", "SWIR2"]
BIMESTRES = [(6, 7), (8, 9), (10, 11), (12, 1), (2, 3), (4, 5)]

def apilado_s2(anio, region):
    """6 medianas bimestrales (jun–may) de S2 L1C armonizado, escenas < 20 % de nubes."""
    col = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
           .filterDate(f"{anio}-06-01", f"{anio + 1}-06-01").filterBounds(region)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).select(B_S2, NOMBRES))
    capas = []
    for k, (m1, m2) in enumerate(BIMESTRES):
        suf = "" if k == 0 else f"_{k}"
        capas.append(col.filter(ee.Filter.calendarRange(m1, m2, "month")).median()
                     .rename([n + suf for n in NOMBRES]))
    return ee.Image.cat(capas)

def mascara_s2(anio, nombre_prov, destino=None):
    """Máscara agrícola 60 m (RF) de la campaña anio/anio+1, entrenada en la provincia."""
    gid = PROVINCIAS[nombre_prov]
    zona = prov.filter(ee.Filter.eq("gid", gid))
    if destino is None:
        destino = deptos.filter(ee.Filter.eq("provincia", nombre_prov))
    pila = apilado_s2(anio, zona)
    puntos = etiqueta.stratifiedSample(numPoints=15000, classBand="b1", region=zona.geometry(),
                                       scale=250, seed=0, geometries=True)
    entren = pila.sampleRegions(collection=puntos, properties=["b1"], scale=60)
    rf = ee.Classifier.smileRandomForest(100).train(entren, "b1", pila.bandNames())
    clas = pila.classify(rf).clip(destino)
    parche = clas.connectedPixelCount(15, False)
    moda = clas.focal_mode(radius=100, kernelType="square", units="meters")
    return (clas.where(parche.lt(4), moda).rename("b1").toByte()
            .set({"campania": anio + 1, "provincia": nombre_prov, "gid": gid}))

# --- control: Rosario 2020/21, código nuevo vs. tu máscara existente --------------
ros = deptos.filter(ee.Filter.eq("link", "82084"))
nueva = mascara_s2(2020, "Santa Fe", destino=ros)
vieja = ee.Image("users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_2020-2021_60m").eq(1).clip(ros)
ha = lambda m: (m.multiply(ee.Image.pixelArea()).divide(1e4)
                .reduceRegion(ee.Reducer.sum(), ros.geometry(), scale=60, crs="EPSG:32720",
                              maxPixels=1e13, tileScale=8).values().get(0))
coinc = nueva.eq(1).And(vieja).multiply(ee.Image.pixelArea()).divide(1e4).reduceRegion(
    ee.Reducer.sum(), ros.geometry(), scale=60, crs="EPSG:32720", maxPixels=1e13, tileScale=8).values().get(0)
a_n, a_v, a_c = ee.List([ha(nueva.eq(1)), ha(vieja), coinc]).getInfo()
print(f"Rosario 2020/21 — nueva: {a_n:,.0f} ha | existente: {a_v:,.0f} ha | "
      f"diferencia {100 * (a_n / a_v - 1):+.1f} % | coincidencia {100 * a_c / a_v:.1f} % de la existente")

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


Colección: Cannot overwrite asset 'projects/earthengine-legacy/assets/users/mnolasco/FORMAR
Rosario 2020/21 — nueva: 101,207 ha | existente: 101,179 ha | diferencia +0.0 % | coincidencia 96.4 % de la existente


In [2]:
CARPETA_MODIS = f"{BASE}/mascaras_modis"
EXT = [10, 11, 12, 15, 16, 18]

# Etiquetas desde la serie MODIS recalculada: > 14 de 22 campañas (2000/01–2021/22)
etiqueta = (ee.ImageCollection([ee.Image(f"{CARPETA_MODIS}/modis_agricola_{y}") for y in range(2001, 2023)])
            .sum().gt(14).rename("b1"))

def mascara_s2(anio, nombre_prov, destino=None, muestreo="provincia"):
    """Máscara agrícola 60 m (RF). muestreo: 'provincia' (método original) o 'deptos'."""
    gid = PROVINCIAS[nombre_prov]
    zona = prov.filter(ee.Filter.eq("gid", gid))
    deptos_prov = deptos.filter(ee.Filter.eq("provincia", nombre_prov))
    if destino is None:
        destino = deptos_prov
    region_muestreo = zona.geometry() if muestreo == "provincia" else deptos_prov.geometry()
    pila = apilado_s2(anio, zona)
    puntos = etiqueta.stratifiedSample(numPoints=15000, classBand="b1", region=region_muestreo,
                                       scale=250, seed=0, geometries=True)
    entren = pila.sampleRegions(collection=puntos, properties=["b1"], scale=60)
    rf = ee.Classifier.smileRandomForest(100).train(entren, "b1", pila.bandNames())
    clas = pila.classify(rf).clip(destino)
    parche = clas.connectedPixelCount(15, False)
    moda = clas.focal_mode(radius=100, kernelType="square", units="meters")
    return (clas.where(parche.lt(4), moda).rename("b1").toByte()
            .set({"campania": anio + 1, "provincia": nombre_prov, "gid": gid, "muestreo": muestreo}))

# --- Prueba La Pampa 2020/21 -------------------------------------------------------
lp = deptos.filter(ee.Filter.eq("provincia", "La Pampa")).select(["link"])
inta = ee.Image(f"{BASE}/MNC_verano-2021")
ext = inta.remap(EXT, [1] * len(EXT), 0).unmask(0)
soja = inta.eq(11).unmask(0)
versiones = {"original": ee.Image("users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_2020-2021_60m").eq(1).unmask(0),
             "provincia": mascara_s2(2020, "La Pampa", muestreo="provincia").eq(1).unmask(0),
             "deptos": mascara_s2(2020, "La Pampa", muestreo="deptos").eq(1).unmask(0)}
bandas = [ext.rename("ext"), soja.rename("soja")]
for k, M in versiones.items():
    bandas += [M.rename(f"{k}"), ext.And(M).rename(f"{k}_ext"), soja.And(M.Not()).rename(f"{k}_sojafuera")]
img_lp = ee.Image.cat(bandas).multiply(ee.Image.pixelArea().divide(1e4))
res_lp = (img_lp.reduceRegions(lp, ee.Reducer.sum(), scale=30, crs="EPSG:32720", tileScale=8)
          .map(lambda f: f.setGeometry(f.geometry().centroid(100))))

tarea = ee.batch.Export.table.toAsset(collection=res_lp, description="prueba_muestreo_lapampa_2021",
                                      assetId=f"{BASE}/prueba_muestreo_lapampa_2021")
tarea.start()
print("Tarea lanzada:", tarea.status()["state"])

Tarea lanzada: READY


In [3]:
import pandas as pd
feats = ee.FeatureCollection(f"{BASE}/prueba_muestreo_lapampa_2021").getInfo()["features"]
info = pd.read_csv("../results/tables/seleccion_imagen_2021.csv", dtype={"link": str})[["link", "departamento"]]
pl = pd.DataFrame([f["properties"] for f in feats])
pl["link"] = pl["link"].astype(str).str.zfill(5)
pl = pl.merge(info, on="link", how="left")

VERS = ["original", "provincia", "deptos"]
def ind(s, k):
    return {"soja_fuera_%": 100 * s[f"{k}_sojafuera"] / s["soja"],
            "mascara_no_agri_%": 100 * (1 - s[f"{k}_ext"] / s[k]),
            "area_vs_INTA_%": 100 * s[k] / s["ext"]}

tot = pl.sum(numeric_only=True)
print("La Pampa (5 departamentos), campaña 2020/21 — total:")
print(pd.DataFrame({k: ind(tot, k) for k in VERS}).T.round(1).to_string())

print("\nÁrea de la máscara respecto del INTA (%), por departamento:")
print(pd.DataFrame({k: pl.apply(lambda r: 100 * r[k] / r["ext"], axis=1) for k in VERS})
      .set_index(pl["departamento"]).round(1).to_string())
print("\nSoja fuera de la máscara (%), por departamento:")
print(pd.DataFrame({k: pl.apply(lambda r: 100 * r[f"{k}_sojafuera"] / r["soja"], axis=1) for k in VERS})
      .set_index(pl["departamento"]).round(1).to_string())

La Pampa (5 departamentos), campaña 2020/21 — total:
           soja_fuera_%  mascara_no_agri_%  area_vs_INTA_%
original            8.2               30.2           130.0
provincia           7.3               27.9           127.4
deptos             26.9                6.4            72.2

Área de la máscara respecto del INTA (%), por departamento:
              original  provincia  deptos
departamento                             
Trenel           115.1      117.2    34.9
Chapaleufú       138.2      132.7    94.9
Maracó           137.7      133.0    90.2
Quemú Quemú      124.9      124.2    64.6
Realicó          125.4      124.2    53.8

Soja fuera de la máscara (%), por departamento:
              original  provincia  deptos
departamento                             
Trenel            24.8       21.9    63.1
Chapaleufú         0.8        0.9     7.8
Maracó             2.2        2.2    13.4
Quemú Quemú       12.7       11.2    38.8
Realicó           12.4       10.7    40.0


In [5]:
PRUEBA = False

trabajos = [(2020, "Santa Fe")] if PRUEBA else \
           [(a, p) for a in range(2019, 2025) for p in PROVINCIAS]      # campañas 2020 … 2025

for anio, p in trabajos:
    gid = PROVINCIAS[p]
    destino = deptos.filter(ee.Filter.eq("provincia", p))
    nombre = f"s2_60m_{anio + 1}_{gid}"
    t = ee.batch.Export.image.toAsset(
        image=mascara_s2(anio, p, muestreo="provincia"), description=nombre,
        assetId=f"{COL_SALIDA}/{nombre}", region=destino.geometry().bounds(),
        scale=60, crs="EPSG:32720", pyramidingPolicy={"b1": "mode"}, maxPixels=1e13)
    t.start()
    print(nombre, t.status()["state"])

s2_60m_2020_24 READY
s2_60m_2020_13 READY
s2_60m_2020_14 READY
s2_60m_2020_22 READY
s2_60m_2020_4 READY
s2_60m_2021_24 READY
s2_60m_2021_13 READY
s2_60m_2021_14 READY
s2_60m_2021_22 READY
s2_60m_2021_4 READY
s2_60m_2022_24 READY
s2_60m_2022_13 READY
s2_60m_2022_14 READY
s2_60m_2022_22 READY
s2_60m_2022_4 READY
s2_60m_2023_24 READY
s2_60m_2023_13 READY
s2_60m_2023_14 READY
s2_60m_2023_22 READY
s2_60m_2023_4 READY
s2_60m_2024_24 READY
s2_60m_2024_13 READY
s2_60m_2024_14 READY
s2_60m_2024_22 READY
s2_60m_2024_4 READY
s2_60m_2025_24 READY
s2_60m_2025_13 READY
s2_60m_2025_14 READY
s2_60m_2025_22 READY
s2_60m_2025_4 READY


In [3]:
# Estado de las tareas recientes y existencia del asset de prueba
for t in ee.batch.Task.list()[:5]:
    s = t.status()
    print(f"{s['description']:<25} {s['state']:<10} {s.get('error_message', '')}")

try:
    info = ee.data.getAsset(f"{COL_SALIDA}/s2_60m_2021_4")
    print("\nAsset de prueba OK:", info["name"])
except Exception as e:
    print("\nAsset de prueba no encontrado:", e)

prueba_muestreo_lapampa_2021 READY      
s2_60m_2025_4             COMPLETED  
s2_60m_2025_22            COMPLETED  
s2_60m_2025_14            COMPLETED  
s2_60m_2025_13            COMPLETED  

Asset de prueba OK: projects/earthengine-legacy/assets/users/mnolasco/FORMAR_2024_2026/mascaras_s2_60m/s2_60m_2021_4


In [4]:
import collections

# Estado de todas las tareas s2_60m_*
estados = collections.Counter()
fallidas = []
for t in ee.batch.Task.list():
    s = t.status()
    if s["description"].startswith("s2_60m_"):
        estados[s["state"]] += 1
        if s["state"] == "FAILED":
            fallidas.append((s["description"], s.get("error_message", "")))
print("Tareas:", dict(estados))
for f in fallidas:
    print("  FALLÓ:", f)

# Assets que efectivamente existen en la colección
col = ee.ImageCollection(COL_SALIDA)
ids = sorted(a["name"].split("/")[-1] for a in ee.data.listAssets({"parent": COL_SALIDA})["assets"])
print(f"\nAssets en la colección: {len(ids)} de 30")
esperados = {f"s2_60m_{a+1}_{g}" for a in range(2019, 2025) for g in PROVINCIAS.values()}
print("Faltan:", sorted(esperados - set(ids)) or "ninguno")

Tareas: {'COMPLETED': 30, 'FAILED': 1}
  FALLÓ: ('s2_60m_2021_4', "Cannot overwrite asset 'projects/earthengine-legacy/assets/users/mnolasco/FORMAR_2024_2026/mascaras_s2_60m/s2_60m_2021_4'.")

Assets en la colección: 30 de 30
Faltan: ninguno


In [5]:
nueva = ee.Image(f"{COL_SALIDA}/s2_60m_2021_4").eq(1)
orig  = ee.Image("users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_2020-2021_60m").eq(1)
region_sf = deptos.filter(ee.Filter.eq("provincia", "Santa Fe")).geometry()

print(ee.Image(f"{COL_SALIDA}/s2_60m_2021_4").toDictionary().getInfo())   # propiedades

areas = (ee.Image.cat(nueva.rename("nueva"), orig.rename("orig"),
                      nueva.And(orig).rename("ambas"), nueva.Or(orig).rename("alguna"))
         .multiply(ee.Image.pixelArea()).divide(1e4)
         .reduceRegion(ee.Reducer.sum(), region_sf, 60, "EPSG:32720",
                       maxPixels=1e13, tileScale=4).getInfo())

print({k: round(v) for k, v in areas.items()})
print(f"Diferencia de área: {100*(areas['nueva']/areas['orig']-1):+.1f} %")
print(f"Coincidencia (ambas/alguna): {100*areas['ambas']/areas['alguna']:.1f} %")

{'campania': 2021, 'gid': 4, 'muestreo': 'provincia', 'provincia': 'Santa Fe'}
{'alguna': 3252056, 'ambas': 3013878, 'nueva': 3099095, 'orig': 3166839}
Diferencia de área: -2.1 %
Coincidencia (ambas/alguna): 92.7 %


In [6]:
for p, gid in PROVINCIAS.items():
    destino = deptos.filter(ee.Filter.eq("provincia", p))
    nombre = f"s2_60m_2019_{gid}"
    t = ee.batch.Export.image.toAsset(
        image=mascara_s2(2018, p, muestreo="provincia"), description=nombre,
        assetId=f"{COL_SALIDA}/{nombre}", region=destino.geometry().bounds(),
        scale=60, crs="EPSG:32720", pyramidingPolicy={"b1": "mode"}, maxPixels=1e13)
    t.start()
    print(nombre, t.status()["state"])

s2_60m_2019_24 READY
s2_60m_2019_13 READY
s2_60m_2019_14 READY
s2_60m_2019_22 READY
s2_60m_2019_4 READY
